In [13]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
import pdfplumber
import os
from google import genai
from dotenv import load_dotenv
import numpy as np

load_dotenv()

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))


In [14]:
pdf_path = r"D:\JEEVA\GENAI\Ai_with_Rag\6th_Social_Science_Term_I_EM - www.tntextbooks.in.pdf"

In [15]:
with pdfplumber.open(pdf_path) as pdf:
    full_text = "\n".join(
        page.extract_text() or ""
        for page in pdf.pages[:]
    )

In [17]:
full_text

'www.tntextbooks.in\nGOVERNMENT OF TAMILNADU\nSTANDARD SIX\nTERM - I\nVOLUME - 3\nSCIENCE\nSOCIAL SCIENCE\nA publication under Free Textbook Programme of Government of Tamil Nadu\nDepartment Of School Education\nUntouchability is Inhuman and a Crime\nwww.tntextbooks.in\nGovernment of Tamil Nadu\nFirst Edition - 2018\nRevised Edition - 2019, 2020, 2022\n(Published under New syllabus in\nTrimester Pattern)\nNOT FOR SALE\nContent Creation\nThe wise\npossess all\nState Council of Educational\nResearch and Training\n© SCERT 2018\nPrinting & Publishing\nTamil NaduTextbook and Educational\nServices Corporation\nwww.textbooksonline.tn.nic.in\nII\nwww.tntextbooks.in\nSOCIAL SCIENCE\n105\nwww.tntextbooks.in\nHOW TO USE\nTHE BOOK\nCase study emphazises\nIt ensures that the\nthe particular part of the\nstudent will attain the\ncontent in a brief and\nlearning skills.\ncrisp manner.\nActivity\n“Do you Know”, “Infobits” and\nActivity helps to\n“Amazing Facts” drive to the\nexperience the content\nef

In [19]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=100,
    separators=["\n\n","\n"," "]
)

text=text_splitter.split_text(full_text)

In [20]:
text
len(text)

139

In [21]:
def convert_embed(text):
    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text
    )
    
    print(result.embeddings[0].values)
    return result.embeddings[0].values

In [22]:
chunk_embedding=[]
for txt in text[:]:
    chunk_embedding.append(convert_embed(text=txt))

[0.016914848, -0.0008886634, 0.021162653, -0.029808028, 0.006817263, 0.0042246557, 0.03124941, -0.0023883644, -0.02748442, 0.0015189194, -0.0024727678, 0.0060122297, 0.011726691, 0.027799359, 0.12991597, 0.007570997, -0.012547, -0.00074570137, -0.0005347388, -0.011533931, 0.016254598, 0.0022385498, -0.0061360477, -0.029117987, -0.00017414402, 0.00015918036, -0.021031395, 0.0075852964, 0.013396822, -0.0023132865, -0.005680694, 0.018155055, 0.005223861, 0.013438267, 0.0050183292, 0.002867511, 0.030828642, -0.015147764, 0.013812893, -0.009130311, 0.0189474, 0.02251999, 0.00262854, -0.0062226667, -0.006366011, -0.008645682, 0.016145801, -0.021437638, -0.0064611565, 0.045348812, 0.0027000986, -0.008218287, -0.007889934, -0.17222169, -0.026161553, -0.04191656, -0.019533535, -0.0047357124, -0.017140057, -0.0073487516, -0.023719056, 0.01441361, -0.010861457, -0.004791774, -0.00046740478, -0.010858677, 0.012387892, 0.0043937834, -0.02581758, 0.01774887, -0.012413198, 0.026118377, -0.018518513, 

In [ ]:
chunk_embedding[0]


In [23]:
type(chunk_embedding[0])

list

In [24]:
type(np.array(chunk_embedding[0]))

numpy.ndarray

In [ ]:
def cosine_similarity(vec1,vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    
    return np.dot(vec1,vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))


In [ ]:
vec1 = convert_embed("worst")
vec2 = convert_embed("bad")
cosine_similarity(vec1,vec2)

In [ ]:
import faiss
import numpy as np

documents = []

def create_faiss_index(dimension): #3072
    return faiss.IndexFlatL2(dimension)

def store_embeddings(index,embeddings,docs):
    global documents
    
    vectors = np.array(embeddings,dtype=np.float32)
    index.add(vectors)
    
    documents.extend(docs)
    
def retrieve_documents(index, query_embedding, top_k=3):
    query_vector = np.array([query_embedding], dtype=np.float32)
    
    distances, indices = index.search(query_vector, top_k)
    
    retrieved_docs = []
    
    for idx in indices[0]:
        if idx != -1:
            retrieved_docs.append(documents[idx])
    return retrieved_docs        
            

In [ ]:
dimensions = len(chunk_embedding[0])
idx = create_faiss_index(dimensions)
store = store_embeddings(idx,chunk_embedding,text[:])

In [ ]:
dimensions

In [ ]:
idx

In [ ]:
store

In [ ]:
text[0]

In [ ]:
query="Translation"
em_query=convert_embed(query)
ret=retrieve_documents(idx,em_query,1)

In [ ]:
ret